# Generate confirmatory chains 

**Notebook is a wrapper around the core script `generate_chains_confirmatory.py` with additional run summaries**

Same structure as the cluster run: a model is served behind an OpenAI-compatible
endpoint by `serve_model.sh`, and the frozen `generate_chains_confirmatory.py`
(`--backend openai_compatible`) drives the chains. The notebook only holds the
parameters and the convenience steps; prompts, cache keys, seed derivation,
parsing, selection, and the edge-case policy all come from the frozen module.

**Start the server in a separate terminal first**:

```
# GPU workstation, tiny model first
./serve_model.sh --model HuggingFaceTB/SmolLM2-135M-Instruct

# 4080, Llama 8-bit (dev only; BF16 does not fit 16 GB)
./serve_model.sh --model hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4 \
  --gpu-mem-util 0.85 --max-model-len 4096

# ASC-5, BF16 on two A100s
./serve_model.sh --model Qwen/Qwen3-32B --tp 2 --max-model-len 32768
```

Prototyping runs on the GPU workstation, not on a CPU-only laptop. The serve
script picks `cpu` automatically only when no GPU is visible; GPU hosts keep
vLLM's default. Point `MODEL_ID` / `BASE_URL` below at the running server and
run the cells.

**Interrupt/resume**: the CLI writes partial chain logs and an append-only JSONL
response cache. Re-running the run cell skips completed hops and replays accepted
calls from cache. This is the frozen rerun semantics, not a notebook approximation.

Edit only the CONSTANTS cell between runs.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys
import urllib.request

GENERATION_DIR = Path.cwd()
if not (GENERATION_DIR / "generate_chains_confirmatory.py").exists():
    # notebook started from the repo root instead of scripts/confirmatory
    GENERATION_DIR = GENERATION_DIR / "scripts" / "generation"
sys.path.insert(0, str(GENERATION_DIR))

import generate_chains_confirmatory as gcc  # for TASK_BANDS + path convention

GENERATOR = GENERATION_DIR / "generate_chains_confirmatory.py"
PYTHON = sys.executable
print("generation dir:", GENERATION_DIR)
print("python:", PYTHON)

generation dir: /home/user/code/echochains-vs-analysis/scripts/confirmatory
python: /home/user/miniconda3/envs/echochain/bin/python


In [2]:
# ---------------------------------------------------------------------
# CONSTANTS: the only cell to edit between runs
# ---------------------------------------------------------------------

# Must match --served-name of the running serve_model.sh process.
# MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
MODEL_ID = "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4"
# MODEL_ID = "Qwen/Qwen3-32B"

BASE_URL = "http://localhost:8892/v1"
# BASE_URL = "http://a471-u099.psy.univie.ac.at:8892/v1"
API_KEY = "local"              # serve_model.sh default

TASK = "paraphrase"          # paraphrase | improve | take_inspiration | expand+compress
CONDITION = "uniform_list"    # direct | uniform_list | vs_weighted

# Explicit decoding pair (works for models outside the pinned vendor table).
# For Llama-3.1-8B use 0.6 / 0.9 (the pinned card settings).
TEMPERATURE = 0.8
TOP_P = 0.9

# Run budget: chains per seed, hops per chain, seeds to use.
# The design uses 5 chains x 200 hops.
N_CHAINS = 5
N_HOPS = 200
N_SEEDS = None                  # None = all seeds in the seed file
REPLICATE = 0

# True = rerun the chains recorded in FAILED_MSG_*.json as a new replicate.
# The CLI chooses the replicate itself (highest failed replicate + 1), so
# REPLICATE is ignored in that mode.
RERUN_FAILED = True # reruns only the failed chains with a new random seed

# Chains run concurrently (independent units are chains, not hops).
# Default 1 = serial. On a 16 GB GPU serving an 8B 4-bit model start at
# 16 and tune against the server log (keep Waiting ~0, KV usage < 70%).
N_WORKERS = 16

# Seed file: empty = the sampler's draw under DATA_DIR
# ($DATA_DIR/data/corpus/sampler/vs_confirmatory_seeds_final.jsonl);
# set to a path for a custom seed file.
SEEDS_FILE = ""

PARTIAL_FLUSH_EVERY = 25     # each flush rewrites the whole chain log;
                             # 25 hops keeps that cheap and loses little
                             # (the response cache makes a replay free)
MAX_RETRIES = 3

# Dev outputs live outside the real data tree.
VS_DATA_ROOT = Path(str(Path.home() / "Dropbox" / "data_export" / "echochains" / "data" / "vs_data"))
LOGS_ROOT = VS_DATA_ROOT / "logs"
CACHE_ROOT = VS_DATA_ROOT / "cache"

print(json.dumps({"model": MODEL_ID, "base_url": BASE_URL, "task": TASK,
                  "condition": CONDITION, "chains": N_CHAINS, "hops": N_HOPS,
                  "temperature": TEMPERATURE, "top_p": TOP_P}, indent=2))

{
  "model": "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
  "base_url": "http://localhost:8892/v1",
  "task": "paraphrase",
  "condition": "uniform_list",
  "chains": 5,
  "hops": 200,
  "temperature": 0.8,
  "top_p": 0.9
}


Server check: is serve_model.sh up and is MODEL_ID the name it exposes?


In [3]:
def list_served(base_url, api_key):
    req = urllib.request.Request(
        base_url.rstrip("/") + "/models",
        headers={"Authorization": f"Bearer {api_key}"})
    with urllib.request.urlopen(req, timeout=10) as resp:
        return json.load(resp)

try:
    served = list_served(BASE_URL, API_KEY)
    names = [m.get("id") for m in served.get("data", [])]
    print("served:", names)
    assert MODEL_ID in names, (
        f"MODEL_ID {MODEL_ID!r} not served; start serve_model.sh with "
        f"--served-name {MODEL_ID} (or change MODEL_ID)")
except Exception as exc:
    raise RuntimeError(
        f"no server at {BASE_URL}; start serve_model.sh in another terminal") from exc

RuntimeError: no server at http://localhost:8892/v1; start serve_model.sh in another terminal

Seeds: default is the sampler's preregistered draw under DATA_DIR.


In [4]:
BAND = gcc.TASK_BANDS[TASK]

if SEEDS_FILE:
    SEEDS_PATH = Path(SEEDS_FILE)
else:
    SEEDS_PATH = (Path(os.environ.get(
        "DATA_DIR", str(Path.home() / "Dropbox" / "data_export" / "echochains")))
        / "data" / "corpus" / "sampler" / "vs_confirmatory_seeds_final.jsonl")

BAND, SEEDS = gcc.load_seed_file(SEEDS_PATH, band=BAND)
assert BAND == gcc.TASK_BANDS[TASK], (
    f"seed band {BAND!r} != task band {gcc.TASK_BANDS[TASK]!r}")
print(f"loaded {len(SEEDS)} seeds from {SEEDS_PATH}")

if False:  # toy-seed fallback kept below for quick no-DATA_DIR tests
    TOY = {
        "sentence": [
            "The committee postponed the vote until the budget office "
            "published its revised projections.",
            "A small crowd gathered at the harbor to watch the last of the "
            "fishing boats return before the storm.",
        ],
        "long": [
            "The committee postponed the vote until the budget office "
            "published its revised projections. Members said the delay was "
            "procedural rather than political, and that the revised numbers "
            "would determine whether the proposal reached the floor at all. "
            "Staff had worked through the weekend to reconcile the two sets "
            "of estimates, which differed by nearly a fifth of the total.",
        ],
    }
    SEEDS = [{"seed_id": i + 1, "text": t, "genre": "prototype"}
             for i, t in enumerate(TOY[BAND])]
    SEEDS_PATH = VS_DATA_ROOT / f"toy_seeds_{BAND}.json"
    SEEDS_PATH.parent.mkdir(parents=True, exist_ok=True)
    SEEDS_PATH.write_text(json.dumps({"band": BAND, "seeds": SEEDS},
                                     ensure_ascii=False, indent=2),
                          encoding="utf-8")
    print(f"wrote {len(SEEDS)} toy {BAND}-band seed(s) to {SEEDS_PATH}")

for s in SEEDS:
    print(s["seed_id"], "|", s["text"][:70].replace("\n", " "), "...")

loaded 90 seeds from /home/user/Dropbox/data_export/echochains/data/corpus/sampler/vs_confirmatory_seeds_final.jsonl
VS_S_ENC_001 | Atlantic had encouraged the inclusion of Crosby into the Stills Nash p ...
VS_S_ENC_002 | Eckhardt was first exposed to the homebrewing of beer by his father, w ...
VS_S_ENC_003 | Since it appeared in the Pokémon series, Celebi has been featured in n ...
VS_S_ENC_004 | Coming off their bye week, Brady and the Patriots traveled to Denver f ...
VS_S_ENC_005 | Despite the fact that these results are not statistically significant, ...
VS_S_ENC_006 | Acts may be drawn up in public or private form, said en minute and en  ...
VS_S_ENC_007 | Ellis subsequently developed a similar process and became a competitor ...
VS_S_ENC_008 | Hofmaier, who is 1.55m tall, was born in Saal an der Donau, the son of ...
VS_S_ENC_009 | On the accession of Edward III his services were rewarded by rapid pro ...
VS_S_ENC_010 | Enzyme structures unfold (denature) when heated or exposed

Progress scan: what is already on disk for this cell?
Uses the module's path convention so it matches what the CLI writes.

In [5]:
def chain_progress(out_dir, seed_id, chain_id, hops):
    """(hops recorded, status) for one chain across every replicate present."""
    sid = f"{seed_id:03d}" if isinstance(seed_id, int) else str(seed_id)
    stem = f"MSG_{sid}_chain_{chain_id:03d}"
    done = 0
    for pattern in (stem + ".json", stem + ".json.partial",
                    stem + "_rep*.json", stem + "_rep*.json.partial"):
        for path in out_dir.glob(pattern):
            log = json.loads(path.read_text(encoding="utf-8"))
            done = max(done, max((e.get("hop", 0) for e in log), default=0))
    return done, ("complete" if done >= hops else "partial" if done else "new")


def scan_progress(task, condition, model, seeds, chains, hops, logs_root):
    out_dir = gcc.cell_out_dir(logs_root, model, task, condition)
    _, failed_pairs = gcc.failed_chains(out_dir)
    failed_set = set(failed_pairs)
    rows = []
    for seed in seeds:
        for chain_id in range(1, chains + 1):
            done, status = chain_progress(out_dir, seed["seed_id"], chain_id,
                                          hops)
            if (status != "complete"
                    and (str(seed["seed_id"]), chain_id) in failed_set):
                status = "failed"  # FAILED_MSG_* record, no complete log
            rows.append((seed["seed_id"], chain_id, done, status))
    return out_dir, rows

selected_seeds = SEEDS if N_SEEDS is None else SEEDS[:N_SEEDS]
out_dir, rows = scan_progress(TASK, CONDITION, MODEL_ID, selected_seeds,
                              N_CHAINS, N_HOPS, LOGS_ROOT)
done_total = sum(r[2] for r in rows)
print(f"cell dir: {out_dir}")
print(f"{done_total}/{len(rows) * N_HOPS} hops already on disk")
for seed_id, chain_id, done, status in rows:
    print(f"  seed {seed_id} chain {chain_id}: {done}/{N_HOPS} hops [{status}]")

failed_rep, failed_pairs = gcc.failed_chains(out_dir)
statuses = {(r[0], r[1]): r[3] for r in rows}
open_failures = [pair for pair in failed_pairs
                 if statuses.get(pair) == "failed"]
if open_failures:
    print(f"\n{len(open_failures)} chain(s) still failed at replicate "
          f"{failed_rep}: {open_failures}")
    print("rerun them with RERUN_FAILED = True (or --rerun-failed)")
elif failed_pairs:
    print(f"\nall {len(failed_pairs)} recorded failed chain(s) have a complete "
          f"rerun; the FAILED_MSG_* records stay as provenance")


cell dir: /home/user/Dropbox/data_export/echochains/data/vs_data/logs/hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4/confirmatory/paraphrase/uniform_list
89150/90000 hops already on disk
  seed VS_S_ENC_001 chain 1: 200/200 hops [complete]
  seed VS_S_ENC_001 chain 2: 200/200 hops [complete]
  seed VS_S_ENC_001 chain 3: 200/200 hops [complete]
  seed VS_S_ENC_001 chain 4: 200/200 hops [complete]
  seed VS_S_ENC_001 chain 5: 200/200 hops [complete]
  seed VS_S_ENC_002 chain 1: 200/200 hops [complete]
  seed VS_S_ENC_002 chain 2: 200/200 hops [complete]
  seed VS_S_ENC_002 chain 3: 200/200 hops [complete]
  seed VS_S_ENC_002 chain 4: 200/200 hops [complete]
  seed VS_S_ENC_002 chain 5: 200/200 hops [complete]
  seed VS_S_ENC_003 chain 1: 200/200 hops [complete]
  seed VS_S_ENC_003 chain 2: 200/200 hops [complete]
  seed VS_S_ENC_003 chain 3: 200/200 hops [complete]
  seed VS_S_ENC_003 chain 4: 200/200 hops [complete]
  seed VS_S_ENC_003 chain 5: 200/200 hops [complete]
  seed VS_S_EN

Run / resume the cell through the frozen CLI. Safe to interrupt and re-run:
completed hops are skipped, accepted calls come from the JSONL cache, and
a chain that exhausts its retry budget is recorded as FAILED_MSG_*.json and
skipped (the run continues); rerun those with --rerun-failed.


In [ ]:
# Run every task x condition cell. A failed chain only skips that chain: the
# cell always attempts its whole work list, so a finished loop means every
# chain in every cell was attempted (unless a nonzero exit is reported below).
cell_results = []
for TASK in ["paraphrase", "improve", "take_inspiration", "expand+compress"]:
    for CONDITION in ["direct", "uniform_list", "vs_weighted"]:
        env = {
            "OPENAI_API_KEY": API_KEY,
            "OPENAI_BASE_URL": BASE_URL,
            "PATH": str(Path(PYTHON).parent) + ":" + os.environ.get("PATH", ""),
            "HOME": os.environ.get("HOME", ""),
        }

        cmd = [PYTHON, str(GENERATOR),
               "--backend", "openai_compatible",
               "--model", MODEL_ID,
               "--task", TASK,
               "--condition", CONDITION,
               "--seeds-file", str(SEEDS_PATH),
               "--chains", str(N_CHAINS),
               "--hops", str(N_HOPS),
               "--partial-flush-every", str(PARTIAL_FLUSH_EVERY),
               "--max-retries", str(MAX_RETRIES),
               "--temperature", str(TEMPERATURE),
               "--top-p", str(TOP_P),
               "--logs-root", str(LOGS_ROOT),
               "--cache-root", str(CACHE_ROOT),
               "--workers", str(N_WORKERS)]
        if RERUN_FAILED:
            # the CLI picks the replicate itself (highest failed + 1)
            cmd += ["--rerun-failed"]
        else:
            cmd += ["--replicate", str(REPLICATE)]
        if N_SEEDS is not None:
            cmd += ["--limit-seeds", str(N_SEEDS)]

        print(" ".join(cmd), "\n")
        result = subprocess.run(cmd, env=env, text=True, capture_output=True)
        print(result.stdout)
        cell_results.append((TASK, CONDITION, result.returncode))
        if result.returncode != 0:
            print(result.stderr)
            raise RuntimeError(
                f"generator exited with {result.returncode} for "
                f"{TASK}/{CONDITION}")

failed_cells = [f"{t}/{c} (exit {rc})" for t, c, rc in cell_results if rc]
print(f"\n{len(cell_results)} cells attempted; "
      + (f"nonzero exits: {failed_cells}" if failed_cells else "all exited cleanly"))


After the loop: what is actually on disk, and which cells still have
unfinished work. A failed chain never stops a cell, so remaining work
means the loop did not reach the end (interrupt/crash) or a cell exited
nonzero. Re-run the run cell to continue; RERUN_FAILED=True recovers
recorded failed chains.


In [8]:
band_counts = {}
for band in sorted(set(gcc.TASK_BANDS.values())):
    try:
        _, band_seeds = gcc.load_seed_file(SEEDS_PATH, band=band)
    except (OSError, ValueError) as exc:
        print(f"note: no seed count for band {band!r}: {exc}")
        band_counts[band] = None
    else:
        band_counts[band] = len(band_seeds)

report = gcc.study_status(LOGS_ROOT, model=MODEL_ID, chains=N_CHAINS,
                          hops=N_HOPS, seeds_by_band=band_counts)
gcc.print_study_status(report)

unfinished = [cell for block in report["models"] for cell in block["cells"]
              if cell["missing"] or cell["incomplete"] or cell["failed"]]
if unfinished:
    print("\ncells with unfinished work:")
    for cell in unfinished:
        print(f"  {cell['task']}/{cell['condition']}: "
              f"{cell['missing']} not started, "
              f"{cell['incomplete']} incomplete, {cell['failed']} failed")
    print("re-run the run cell to continue (plain resume); "
          "RERUN_FAILED=True recovers the failed chains.")
else:
    print("\nall cells complete")


logs root: /home/user/Dropbox/data_export/echochains/data/vs_data/logs

model: hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4
  paraphrase/direct         450/450   failed 0   incomplete 0  
  paraphrase/uniform_list   445/450   failed 5   incomplete 0  
  paraphrase/vs_weighted    450/450   failed 0   incomplete 0  
  improve/direct         450/450   failed 0   incomplete 0  
  improve/uniform_list   361/450   failed 89  incomplete 0  
  improve/vs_weighted    446/450   failed 4   incomplete 0  
  take_inspiration/direct         450/450   failed 0   incomplete 0  
  take_inspiration/uniform_list   449/450   failed 1   incomplete 0  
  take_inspiration/vs_weighted    448/450   failed 2   incomplete 0  
  expand+compress/direct         450/450   failed 0   incomplete 0  
  expand+compress/uniform_list   374/450   failed 76  incomplete 0  
  expand+compress/vs_weighted    439/450   failed 11  incomplete 0  
  total                      5212/5400  failed 188 incomplete 0  
  failed: pa

Inspect chains

In [ ]:
condition = "uniform_list"
task = "improve"
repo = "hugging-quants"
model = "Meta-Llama-3.1-8B-Instruct-AWQ-INT4"

log_content = sorted(os.listdir(Path(LOGS_ROOT, repo, model, "confirmatory", task, condition)))
len(log_content)

In [ ]:
success = [x for x in log_content if (x.startswith("MSG_") and (x.endswith(".json")))]

len(success)

In [ ]:
partial = [x for x in log_content if x.endswith(".json.partial")]

len(partial)

In [ ]:
failed = [x for x in log_content if x.startswith("FAILED")]

len(failed)

In [ ]:
paused = [x for x in log_content if x.startswith(".paused")]

len(paused)

In [ ]:
rest = [x for x in log_content if (x not in success) and (x not in partial) and (x not in failed) and (x not in paused)]
rest

In [ ]:
with open(Path(LOGS_ROOT, repo, model, "confirmatory", task, condition,failed[0]), "r") as fp:
    failed_chain = json.load(fp)

In [ ]:
failed_chain.keys()

In [ ]:
failed_chain["failed_hop"]

In [ ]:
len(failed_chain["failed_raw_responses"])

In [ ]:
len(failed_chain["hop_log"])

In [ ]:
failed_chain["hop_log"][-1].keys()

In [ ]:
failed_chain["hop_log"][-1]["candidates"]

In [ ]:
print(failed_chain["failed_raw_responses"][0])

In [ ]:
raw_r = {
  "improved_texts": [
    {
      "text": "The CARR Assessment: A Revolutionary Diagnostic Tool\nThe CARR assessment is a groundbreaking diagnostic tool that empowers researchers to identify the underlying causes and core characteristics of incurable and untreatable conditions."
    },
    {
      "text": "Revolutionizing Research: The Power of the CARR Assessment\nThis innovative diagnostic tool enables researchers to pinpoint the underlying causes and core characteristics of incurable and untreatable conditions, paving the way for targeted interventions and improved patient outcomes."
    },
    {
      "text": "Unlocking New Possibilities: The CARR Assessment in Action\nBy leveraging the CARR assessment, researchers can develop targeted interventions to enhance the quality of life for those affected, drive innovation in the field, and stimulate research discovery, ultimately leading to better patient outcomes and improved treatment options."
    },
    {
      "text": "Transforming Lives with the CARR Assessment\nThe CARR assessment is a game-changing diagnostic tool that allows researchers to identify the underlying causes of incurable and untreatable conditions, opening up new possibilities for targeted interventions and improved patient outcomes."
    },
    {
      "text": "Advancing Research with the CARR Assessment: A Diagnostic Breakthrough\nThis diagnostic tool empowers researchers to pinpoint the underlying causes and core characteristics of incurable and untreatable conditions, driving innovation in the field and stimulating research discovery, ultimately leading to better patient outcomes and improved treatment options."
    }
  ]
}